# w2v-bert-2.0 karaoke finetune

In [ ]:
import json
import re
import unicodedata
from dataclasses import dataclass
from math import ceil

import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import login as hf_login
from transformers import (
    EarlyStoppingCallback,
    EvalPrediction,
    SeamlessM4TFeatureExtractor,
    Trainer,
    TrainingArguments,
    Wav2Vec2BertForCTC,
    Wav2Vec2BertProcessor,
    Wav2Vec2CTCTokenizer,
)
from transformers.modeling_outputs import CausalLMOutput

## Load and normalize dataset

In [ ]:
hf_login()

In [ ]:
dataset = load_dataset("NextFire/karaoke-alignments", split="train")


def normalize_uroman(text: str):
    text = text.casefold()
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    text = re.sub("[^a-z ]", " ", text)
    text = re.sub(" +", " ", text)
    return text


def normalize_dataset(example):
    morae = example["morae"]
    text = ""
    for m in morae:
        norm = normalize_uroman(m["mora"])
        if not norm.strip():
            morae = None
            break
        m["mora"] = norm
        text += norm
    return {"text": text, "morae": morae}


dataset = dataset.map(normalize_dataset, num_proc=8)
dataset = dataset.filter(bool, input_columns="morae", num_proc=8)

## Build vocabulary

In [ ]:
vocab = dataset.map(
    lambda batch: {"vocab": list(set("".join(batch)))},
    input_columns="text",
    remove_columns=dataset.column_names,
    batched=True,
    batch_size=None,
    keep_in_memory=True,
)["vocab"]

vocab_dict = {v: k for k, v in enumerate(sorted(vocab))}
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

with open("vocab.json", "w") as f:
    json.dump(vocab_dict, f)

## Transformers items

In [ ]:
@dataclass
class DataCollatorForcedAlignmentWithPadding:
    processor: Wav2Vec2BertProcessor
    padding: bool = True

    def __call__(self, features) -> dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.pad(
            input_features=input_features, padding=self.padding, return_tensors="pt"
        )
        assert batch

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.pad(
            labels=label_features, padding=self.padding, return_tensors="pt"
        )
        assert labels_batch
        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels

        return batch

In [ ]:
# type: ignore
class Wav2Vec2BertForForcedAlignment(Wav2Vec2BertForCTC):
    """Wav2Vec2Bert with frame-level CE loss instead of CTC for forced alignment."""

    def forward(
        self,
        input_features: torch.Tensor | None,
        attention_mask: torch.Tensor | None = None,
        output_attentions: bool | None = None,
        output_hidden_states: bool | None = None,
        return_dict: bool | None = None,
        labels: torch.Tensor | None = None,
        **kwargs,
    ):
        outputs = super().forward(
            input_features=input_features,
            attention_mask=attention_mask,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
            labels=None,  # skip CTC loss
            **kwargs,
        )
        if labels is not None:
            logits = outputs.logits  # (batch, time, vocab)
            T = min(logits.size(1), labels.size(1))
            loss = F.cross_entropy(
                logits[:, :T].contiguous().view(-1, logits.size(-1)),
                labels[:, :T].contiguous().view(-1),
                ignore_index=-100,
            )
            outputs = CausalLMOutput(
                loss=loss,
                logits=outputs.logits,
                hidden_states=outputs.hidden_states,
                attentions=outputs.attentions,
            )
        return outputs

In [ ]:
feature_extractor = SeamlessM4TFeatureExtractor.from_pretrained("facebook/w2v-bert-2.0")
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    "./", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|"
)
processor = Wav2Vec2BertProcessor(
    feature_extractor=feature_extractor, tokenizer=tokenizer
)
data_collator = DataCollatorForcedAlignmentWithPadding(processor=processor, padding=True)
model = Wav2Vec2BertForForcedAlignment.from_pretrained(
    "facebook/w2v-bert-2.0",
    attention_dropout=0.1,
    hidden_dropout=0.1,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.05,
    ctc_loss_reduction="mean",
    add_adapter=False,
    pad_token_id=processor.tokenizer.pad_token_id,  # type: ignore
    vocab_size=len(processor.tokenizer),  # type: ignore
)

## Prepare dataset

In [ ]:
# Encoder outputs at 50Hz (20ms/frame); adapter downsamples by adapter_stride when add_adapter=True
adapter_stride = model.config.adapter_stride if model.config.add_adapter else 1
ms_per_frame = 20 * adapter_stride


def prepare_dataset(batch):
    audio = batch["audio"]
    features = processor(
        audio=audio["array"],
        sampling_rate=audio["sampling_rate"],  # type: ignore
    ).input_features[0]
    batch["input_features"] = features
    batch["input_length"] = len(features)

    # Frame-level labels from morae timings
    num_output_frames = ceil(len(features) / adapter_stride)
    frame_labels = [processor.tokenizer.pad_token_id] * num_output_frames  # type: ignore
    for mora in batch["morae"]:
        token_ids = processor.tokenizer.encode(mora["mora"])  # type: ignore
        n_tokens = len(token_ids)
        start_frame = mora["start"] // ms_per_frame
        end_frame = mora["end"] // ms_per_frame
        n_frames = end_frame - start_frame
        if n_tokens > 0 and n_frames > 0:
            for i, token_id in enumerate(token_ids):
                t_start = start_frame + (i * n_frames) // n_tokens
                t_end = start_frame + ((i + 1) * n_frames) // n_tokens
                for f in range(t_start, min(t_end, num_output_frames)):
                    frame_labels[f] = token_id

    batch["labels"] = frame_labels

    return batch


dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset.column_names,
    num_proc=8,
    new_fingerprint=f"prepared_dataset_{adapter_stride=}",
)
dataset = dataset.shuffle(seed=42)
split = dataset.train_test_split(test_size=0.1, seed=42)

## Train

In [ ]:
def compute_metrics(eval_pred: EvalPrediction):
    pred_logits = eval_pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    label_ids = eval_pred.label_ids

    # Frame-level accuracy (ignoring padding)
    mask = label_ids != -100
    frame_accuracy = ((pred_ids == label_ids) & mask).sum() / mask.sum()  # type: ignore

    # Boundary MAE: compare token transition positions
    pad_id = processor.tokenizer.pad_token_id  # type: ignore
    boundary_errors = []
    for pred_id, label_id in zip(pred_ids, label_ids):
        valid = label_id != -100
        pred_valid, label_valid = pred_id[valid], label_id[valid]
        pred_bounds = [
            i
            for i in range(1, len(pred_valid))
            if pred_valid[i] != pred_valid[i - 1]
            and pred_valid[i] != pad_id
            and pred_valid[i - 1] != pad_id
        ]
        label_bounds = [
            i
            for i in range(1, len(label_valid))
            if label_valid[i] != label_valid[i - 1]
            and label_valid[i] != pad_id
            and label_valid[i - 1] != pad_id
        ]
        for bound in label_bounds:
            if pred_bounds:
                nearest = min(pred_bounds, key=lambda x: abs(x - bound))
                boundary_errors.append(abs(nearest - bound) * ms_per_frame)

    boundary_mae_ms = float(np.mean(boundary_errors)) if boundary_errors else 0.0

    return {
        "frame_accuracy": float(frame_accuracy),
        "boundary_mae_ms": boundary_mae_ms,
    }


training_args = TrainingArguments(
    output_dir="w2v-bert-2.0-ForcedAligner-karaoke-ja-Latn",
    # push_to_hub=True,
    num_train_epochs=10,
    per_device_train_batch_size=48,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=48,
    eval_accumulation_steps=1,
    gradient_checkpointing=True,
    bf16=True,
    tf32=True,
    dataloader_pin_memory=True,
    dataloader_num_workers=4,
    lr_scheduler_type="cosine",
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_steps=0.05,
    eval_strategy="steps",
    eval_steps=0.05,
    save_steps=0.05,
    save_total_limit=5,
    metric_for_best_model="boundary_mae_ms",
    greater_is_better=False,
    load_best_model_at_end=True,
    logging_steps=10,
    report_to="trackio",
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    processing_class=processor.feature_extractor,  # type: ignore
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

In [ ]:
trainer.train()

In [ ]:
processor.save_pretrained("w2v-bert-2.0-ForcedAligner-karaoke-ja-Latn")
trainer.save_model("w2v-bert-2.0-ForcedAligner-karaoke-ja-Latn")